# Serve MusPsy (or base Llama-3-8B) via vLLM on RunPod

This notebook starts an OpenAI-compatible inference server on the GPU pod using vLLM's own native `vllm serve` CLI directly (per vLLM's [installation](https://docs.vllm.ai/en/latest/getting_started/installation/gpu/) and [quickstart](https://docs.vllm.ai/en/latest/getting_started/quickstart/) docs), rather than routing through LLaMA-Factory's Python wrapper.

**Why not LLaMA-Factory's `llamafactory-cli api` (the earlier approach)?** It works, but forces vLLM's dependencies and LLaMA-Factory's dependencies (transformers, peft, accelerate, huggingface-hub, typing_extensions, torchaudio/torchvision - all with their own overlapping-but-not-identical version constraints) into one environment, which caused a long chain of version conflicts. It also has its own fragile internal `vllm.__version__` parsing that breaks if that metadata ever gets corrupted. None of that machinery is actually needed just to serve a LoRA adapter - vLLM does this natively. LLaMA-Factory is still the right tool for *training* (see `muspsy_finetune.ipynb`); this notebook only needs vLLM itself.

Your local machine (running `thesis-framework`) then points `--counselor_model` at this server's URL instead of Gemini/DeepSeek/OpenAI, so all the actual GPU inference happens here on RunPod.

**Toggle**: set `SERVE_MODE` below to `"muspsy"` (base model + your trained LoRA adapter) or `"base"` (plain Llama-3-8B-Instruct, no adapter) — the two conditions needed for a fair framework-vs-MusPsy comparison on the same backbone. Only one server runs at a time; switch by stopping (Step 6) and re-launching (Step 4) with a different `SERVE_MODE`.

**Cost note**: the pod bills by the hour whether or not you're sending requests — stop the server (and the pod) when you're done testing.

## Step 0: Check CUDA works before installing anything
Cheap to check now (seconds), expensive to discover after a multi-minute install. This session hit the same failure repeatedly on certain Community Cloud pods: `nvidia-smi` reports a perfectly healthy GPU, but `torch.cuda.is_available()` returns `False` — traced to `CUDA_VISIBLE_DEVICES` being set to an empty string, which hides all devices from the CUDA runtime (a different code path than `nvidia-smi`'s, which uses NVML and isn't affected). Secure Cloud pods resolved it reliably; if this check fails, try a different pod before spending time on Step 1.

In [ ]:
!nvidia-smi

try:
    import torch
    print("torch already installed:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    assert torch.cuda.is_available(), (
        "CUDA not available even though nvidia-smi above may show a healthy GPU. If "
        "CUDA_VISIBLE_DEVICES is set to an empty string, that's the cause (echo it to check). "
        "Try a different pod - Secure Cloud resolved this reliably - before proceeding to Step 1."
    )
    print("CUDA check passed - safe to proceed to Step 1.")
except ImportError:
    print("torch not installed yet on this pod - will be installed fresh in Step 1 "
          "(`uv pip install vllm --torch-backend=auto`). Re-run this cell after Step 1 "
          "to confirm CUDA works before launching the server.")

## Step 1: Install vLLM
Pinning to `vllm==0.9.2` rather than the latest unpinned release. This isn't a workaround for our own environment - it's a **confirmed upstream packaging bug** ([vllm-project/vllm#43435](https://github.com/vllm-project/vllm/issues/43435), closed by maintainers as "not planned"): the latest vLLM PyPI wheel's compiled `_C_stable_libtorch` extension references `libcudart.so.13` regardless of which `--torch-backend` you request, because `--torch-backend` only controls which *torch* wheel gets pulled, not which *vLLM* wheel does - vLLM's own binary is fixed at publish time. `0.9.2` predates this regression and requires `torch==2.7.0` (still satisfies `torch.library.infer_schema`, which needs torch>=2.5.0). `--torch-backend=auto` lets uv correctly match torch to what 0.9.2 actually needs.

In [ ]:
!pip install -U uv
# Pinned to vllm==0.9.2 specifically - see markdown above: the latest vLLM release has
# a confirmed upstream bug (vllm-project/vllm#43435) where its compiled binary needs
# libcudart.so.13 no matter what --torch-backend you specify. 0.9.2 predates it.
#
# torch==2.7.0 is an explicit direct target here too (vllm 0.9.2's own declared exact
# requirement), not left as an implicit transitive dependency - a pod disk reset (e.g.
# after editing exposed ports via Stop/Edit/Start) can put you back on the base image's
# torch 2.4.1+cu124, and --reinstall on vllm alone doesn't reliably force torch to be
# re-resolved in the same pass. Pinning both explicitly in one command does.
!uv pip install --system --reinstall "vllm==0.9.2" "torch==2.7.0" --torch-backend=auto

# vllm==0.9.2 only declares a transformers FLOOR (>=4.51.1), no ceiling, so uv pulls
# whatever transformers is newest today. Newer transformers releases added native
# support for the "aimv2" vision model config - which collides with vllm 0.9.2's own
# bundled compatibility shim (transformers_utils/configs/ovis.py) that tries to
# register "aimv2" itself, assuming it doesn't already exist. Confirmed as the same
# known collision in vllm-ascend#2046 (no upstream fix, only known workaround is
# pinning transformers to a pre-collision version). 4.51.1 is vllm 0.9.2's own
# declared floor - the version it was actually built/tested against.
!uv pip install --system --reinstall "transformers==4.51.1"

# Same pattern again: nothing pins a numpy ceiling, so uv grabs the newest (2.4.x),
# but numba (used by vllm's optional speculative-decoding n-gram proposer, imported
# unconditionally at worker startup) only supports numpy<=2.2. Pin numpy down rather
# than trying to force numba to a newer release, since numpy 2.2 is plenty for this
# inference-only workload.
!uv pip install --system --reinstall "numpy<2.3"

import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available - check `nvidia-smi`'s CUDA Version, and confirm you're on a healthy pod (see muspsy_finetune.ipynb notes on Community Cloud CUDA_VISIBLE_DEVICES issues)."

from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.5.0"), (
    f"torch {torch.__version__} is too old (needs >=2.5.0 for torch.library.infer_schema). "
    "If this happens right after a pod Stop/Edit/Start cycle, the container disk was likely "
    "reset - this cell is safe to just re-run from scratch."
)

import transformers
print("transformers:", transformers.__version__)
assert transformers.__version__ == "4.51.1", f"transformers pin failed, got {transformers.__version__!r}"

import numpy
print("numpy:", numpy.__version__)
assert Version(numpy.__version__) < Version("2.3"), f"numpy pin failed, got {numpy.__version__!r}"

import vllm
print("vllm:", vllm.__version__)
assert vllm.__version__ == "0.9.2", f"vllm pin failed, got {vllm.__version__!r}"

## Step 2: Choose what to serve
`"muspsy"` — base Llama-3-8B-Instruct + your trained LoRA adapter (the fine-tuned counselor), served under the model name `muspsy`.
`"base"` — plain Llama-3-8B-Instruct, no adapter (the fairness-comparison condition: same backbone your framework's baselines would use).

`ADAPTER_CHECKPOINT` accepts either a **Hugging Face Hub repo ID** (e.g. `your-username/llama3-8b-muspsy`, matching `HF_HUB_MODEL_ID` from the training notebook's `PUSH_TO_HUB` step - vLLM downloads it the same way it downloads the base model) or a **local path** (e.g. `saves/llama3-8b/lora/muspsy`, only if this pod also ran training). If the Hub repo is private, make sure `HF_TOKEN` (Step 3) has read access to it.

In [ ]:
# Backbone choice, matching muspsy_finetune.ipynb's BASE_MODEL_CHOICE toggle. vllm==0.9.2
# (pinned in Step 1) already has Qwen3ForCausalLM registered (verified against the v0.9.2
# tag directly), so no changes were needed to the install cell for this switch.
#
# Switched back to "llama3-8b" from "qwen3-8b" after discovering a Qwen3-8B-backbone-
# specific regression: the exact same LoRA training recipe (same data, DATASET_VARIANT=
# "fixed") reliably reproduces Task 1's full 5-field structured memory format on
# Llama-3-8B (confirmed directly against pilot20_final_20260709_180745's trace), but
# consistently collapses to a bare "Counseling Summary:" on Qwen3-8B - ruled out prompt/
# input-format mismatch and token-budget truncation (tested up to max_tokens=2048) as
# causes, so this is backbone-specific, not fixable from the serving/prompt side. The
# original motivation for Qwen3-8B (more context headroom) applies to the framework's
# OWN baseline agents tested via SERVE_MODE="base" below, not to MusPsy itself - MusPsy's
# own training data averages ~5693 tokens/client (paper Table 1), comfortably inside
# Llama-3's 8192 native context.
BASE_MODEL_CHOICE = "llama3-8b"  # "llama3-8b" or "qwen3-8b"
assert BASE_MODEL_CHOICE in ("llama3-8b", "qwen3-8b")
_BASE_MODELS = {
    "llama3-8b": "meta-llama/Meta-Llama-3-8B-Instruct",
    "qwen3-8b": "Qwen/Qwen3-8B",
}
BASE_MODEL = _BASE_MODELS[BASE_MODEL_CHOICE]

SERVE_MODE = "muspsy"  # "muspsy" or "base"
# Matches muspsy_finetune.ipynb's HF_HUB_MODEL_ID = f"thanaphatt1/{BASE_MODEL_CHOICE}-muspsy-{DATASET_VARIANT}"
ADAPTER_CHECKPOINT = f"thanaphatt1/{BASE_MODEL_CHOICE}-muspsy-fixed"  # HF Hub repo ID, or a local path if this pod also ran training

API_PORT = 8000
API_KEY = "muspsy-dev-key"  # shared bearer token thesis-framework will send - change this to something private

# max_model_len depends on SERVE_MODE, not just the backbone - these protect against
# two different things:
#   "muspsy": 6144, deliberately conservative - protects the LoRA adapter's narrow
#             training distribution (99th percentile ~2280 tokens, max ~3690 - see
#             task3_data_quality_finding.md). Going far beyond that risks the silent
#             quality drift discussed when this number was first chosen, regardless
#             of what the backbone could technically support.
#   "base":   no adapter to protect, so this uses each backbone's actual native
#             context instead - this is the condition your framework's own baseline
#             agents (rich CC/DAP/multi-session prompts) run under via
#             --counselor_model vllm:{BASE_MODEL_CHOICE}-base, and the whole point of
#             moving off Llama-3 (8192 native) to Qwen3-8B was more headroom here.
_BASE_MODE_MAX_LEN = {"llama3-8b": 8192, "qwen3-8b": 32768}
max_model_len = 6144 if SERVE_MODE == "muspsy" else _BASE_MODE_MAX_LEN[BASE_MODEL_CHOICE]

serve_cmd = [
    "vllm", "serve", BASE_MODEL,
    "--port", str(API_PORT),
    "--api-key", API_KEY,
    "--max-model-len", str(max_model_len),
    "--gpu-memory-utilization", "0.85",
    "--dtype", "bfloat16",
]
# NOTE: --default-chat-template-kwargs was tried here as a server-side belt-and-
# suspenders for Qwen3's enable_thinking, but vllm==0.9.2 (pinned - see Step 1) doesn't
# recognize that flag; it was only added in a later vLLM release. Not upgrading vLLM
# just for this - it would reopen the libcudart.so.13 packaging bug (vllm-project/vllm
# #43435) that 0.9.2 was specifically pinned to avoid. Thinking is still reliably
# disabled via the per-request path: thesis-framework/utils/llm_utils.py's get_llm()
# vllm: branch sets extra_body.chat_template_kwargs.enable_thinking=False on every
# request, which is the mechanism actually exercised by every test run so far.

# served_model_name must match thesis-framework/utils/llm_utils.py's get_llm() vllm:
# branch, which expects "muspsy" or "<backbone>-base" (get_llm("vllm:muspsy") /
# get_llm(f"vllm:{BASE_MODEL_CHOICE}-base")) - --served-model-name aliases the base
# model to a short name instead of forcing API requests to use its full HF Hub path.
if SERVE_MODE == "muspsy":
    served_model_name = "muspsy"
    serve_cmd += [
        "--enable-lora",
        "--lora-modules", f"muspsy={ADAPTER_CHECKPOINT}",
        "--max-lora-rank", "32",  # must be >= the LoRA rank used in training
    ]
elif SERVE_MODE == "base":
    served_model_name = f"{BASE_MODEL_CHOICE}-base"
    serve_cmd += ["--served-model-name", served_model_name]
else:
    raise ValueError(f"Unknown SERVE_MODE: {SERVE_MODE!r}")

print(f"BASE_MODEL_CHOICE={BASE_MODEL_CHOICE!r} ({BASE_MODEL!r}), SERVE_MODE={SERVE_MODE!r}, served model name for API requests: {served_model_name!r}")
print(f"max_model_len: {max_model_len}")
print(f"ADAPTER_CHECKPOINT: {ADAPTER_CHECKPOINT!r}" if SERVE_MODE == "muspsy" else "(no adapter - vanilla base model)")
print("Command:", " ".join(serve_cmd))

## Step 3: Set your Hugging Face token
Needed to download `Meta-Llama-3-8B-Instruct` (gated model) and, for `SERVE_MODE="muspsy"`, the adapter repo if it's private.

In [ ]:
%env HF_TOKEN=hf_your_read_or_write_token_here

## Step 4: Launch the server in the background
`vllm serve` blocks forever while serving, so it's launched as a background process here (not with a blocking `!` cell) — this lets you keep using the notebook to health-check and test it, and to stop it cleanly later.

In [ ]:
import subprocess, os

env = os.environ.copy()
# vLLM's internal EngineCore worker process defaults to fork(), which fails with
# "Cannot re-initialize CUDA in forked subprocess" whenever CUDA has already been
# touched in the parent process before the fork happens. spawn starts a fresh
# interpreter instead, avoiding the inherited CUDA context.
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

log_file = open("vllm_server.log", "w")
server_proc = subprocess.Popen(
    serve_cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"Server starting in background, PID={server_proc.pid}. Logs: vllm_server.log")

## Step 5: Wait for the server to become ready
Loading the 8B model + vLLM engine init typically takes a couple of minutes. This polls until the `/v1/models` endpoint responds.

In [ ]:
import time, requests

url = f"http://localhost:{API_PORT}/v1/models"
headers = {"Authorization": f"Bearer {API_KEY}"}

for attempt in range(60):
    if server_proc.poll() is not None:
        raise RuntimeError(
            f"Server process exited early (code {server_proc.returncode}). Check vllm_server.log for the traceback."
        )
    try:
        resp = requests.get(url, headers=headers, timeout=5)
        if resp.status_code == 200:
            print("Server is up.")
            print(resp.json())
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    raise TimeoutError("Server did not become ready in time. Check vllm_server.log.")

## Step 6: Sanity-check with a sample request
Confirms the OpenAI-compatible chat endpoint actually works before wiring up `thesis-framework`. Note the `model` field must be `served_model_name` from Step 2 (`"muspsy"` for the LoRA adapter, or the exact base model ID for `SERVE_MODE="base"`) - vLLM's OpenAI server matches requests against the served model name(s), not an arbitrary string.

In [ ]:
chat_url = f"http://localhost:{API_PORT}/v1/chat/completions"
payload = {
    "model": served_model_name,
    "messages": [
        {"role": "user", "content": "Hi, I've been feeling anxious about an upcoming exam. Can you help?"}
    ],
    "temperature": 0.7,
    "max_tokens": 300,
}
resp = requests.post(chat_url, headers={**headers, "Content-Type": "application/json"}, json=payload, timeout=60)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])

## Step 7: Expose the port to your local machine
In the RunPod dashboard: open this pod's **Connect** panel -> **HTTP Service** (or **TCP Port Mapping** on Secure Cloud) -> expose port `8000`. RunPod gives you a public URL like:

```
https://<pod-id>-8000.proxy.runpod.net
```

From `thesis-framework` on your PC, this is the `base_url` for a new `get_llm()` branch pointed at this server (append `/v1`), with the `API_KEY` set above as the bearer token, and `served_model_name` (`"muspsy"` or the base model ID) as the model string. Requests sent from your PC travel to this URL; all GPU inference happens here on the pod.

## Step 8: Stop the server when done
Run this before switching `SERVE_MODE` and re-launching (Step 4), or before shutting down the pod. Remember the pod itself keeps billing until you stop/terminate it separately in the RunPod dashboard.

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=30)
print("Server stopped.")